# 실습 4-1 : 결측치 및 이상치 처리

#### **<실습 내용>**

1. 결측치 처리
- 결측치 확인
- 결측치 제거
- 결측치 대치 (값 기반, 통계 기반, 머신러닝 기반 (KNN Imputer), 시계열 보간법)

2. 이상치 탐지
- 이상치 확인
- IQR 기반 이상치 제거

## 분석 준비

### 주요 라이브러리 호출

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split

### 데이터 불러오기

In [ ]:
data = pd.read_csv('dataset/day4-1_data.csv')
print("데이터 크기:", data.shape)

data.head()

---

## 1) 결측치 처리

> **결측치 처리** 방법으로는 대표적으로 두 가지 방식이 있음
> 1. **제거**: 결측치가 존재하는 입력변수 또는 행을 제거
> 2. **대치 (보간)**: 결측치를 특정 값으로 대체

### 1-1) 결측치 확인

In [ ]:
# 데이터의 각 위치가 결측치인지 확인
data.isnull()

In [ ]:
# 변수별 결측치 개수 확인
data.isnull().sum()

In [ ]:
# 실무에서 자주 사용하는 결측치 확인 코드

# 1. 데이터 전체 결측치 개수 / 비율 한눈에 보기
print("전체 결측치 개수:", data.isnull().sum().sum())

In [ ]:
# 2. 변수별 결측치 비율(%)
print(data.isnull().mean() * 100)

In [ ]:
# 3. 결측치 비율 높은 순 정렬

# .sort_values() : 값을 기준으로 정렬해주는 함수 
# ascending=False : 내림차순(큰 값 -> 작은 값)으로 정렬 (True면 오름차순)

print(data.isnull().mean().sort_values(ascending=False))

In [ ]:
# 4. 결측치가 존재하는 변수만 보기
print(data.isnull().sum()[data.isnull().sum() > 0])

In [ ]:
# 5. 결측치 여부에 따른 변수 개수 확인
print("결측치 있는 변수 수:", (data.isnull().sum() > 0).sum())
print("결측치 없는 변수 수:", (data.isnull().sum() == 0).sum())

In [ ]:
# 6. 결측치 요약 표 (DataFrame 형태)

missing_summary = pd.DataFrame({
    "결측개수": data.isnull().sum(),               # 컬럼별 결측치 개수
    "결측비율(%)": data.isnull().mean() * 100,     # 컬럼별 결측치 비율(%)
    "dtype": data.dtypes                          # 컬럼별 데이터 타입 (대치 방법 선택에 참고)
}).query("결측개수 > 0").sort_values("결측비율(%)", ascending=False)                            
   # 결측치가 1개 이상인 컬럼만 남기고 결측 비율이 높은 컬럼부터 정렬

missing_summary

### 1-2) 결측치 제거

> **결측치 제거** 방법을 선택하기 전, 행을 제거할지 변수를 제거할지 먼저 판단해야 함
>
> **판단 기준**
> - 결측 있는 행의 비율이 낮고(예: 5% 미만) 데이터가 충분한 경우 → **행 제거**
> - 특정 변수의 결측 비율이 매우 높은 경우(예: 50% 이상) → 해당 **변수(열) 제거**

In [ ]:
# 행을 제거할지 변수를 제거할지 판단하기

# 변수 기준 결측 비율 확인 → 비율이 매우 높은 변수가 있으면 "열 제거" 고려
col_missing_ratio = data.isnull().mean().sort_values(ascending=False)

# 결측 비율이 50%를 넘는 변수 개수 확인
n_high_missing = (col_missing_ratio >= 0.5).sum()
print(f"결측 비율 50% 이상인 변수 개수: {n_high_missing}개")

In [ ]:
# 행 기준 결측 비율 확인 → 결측 있는 행의 비율이 작으면 "행 제거"로도 데이터 손실이 적음
row_missing_ratio = data.isnull().any(axis=1).mean()
print(f"결측치가 하나라도 있는 행의 비율: {row_missing_ratio * 100:.2f}%")

In [ ]:
# 결측 비율 50% 이상인 변수 이름 뽑기
cols_to_drop = col_missing_ratio[col_missing_ratio >= 0.5].index

# 해당 변수들을 제거
data2 = data.drop(columns=cols_to_drop)

print("변수 제거 전:", data.shape[1], "-> 변수 제거 후:", data2.shape[1])

### 1-3) 결측치 대치

> **결측치 대치(Single Imputation)** 는 결측치를 하나의 값으로 보완하는 방법임

| 대치 방법 | 사용 시점 |
|---|---|
| 0 | 결측이 "없음"을 의미하는 경우 |
| 매우 큰 값 (99999 등) | 결측 자체를 하나의 신호로 활용하고자 하는 경우 |
| 평균(Mean) | 연속형 변수의 대표적인 대치 방법 |
| 중앙값(Median) | 이상치가 많은 연속형 변수 |
| 최빈값(Mode) | 범주형 변수의 대표적인 대치 방법 |
| Unknown | 결측 자체를 하나의 범주로 취급하고자 하는 경우 |

In [ ]:
# 0으로 보간
data_zero = data2.fillna(0)
print("[0으로 보간] 결측치 수:", data_zero.isna().sum().sum())

In [ ]:
# 매우 큰 값으로 보간
data_large = data2.fillna(99999)
print("[99999로 보간] 결측치 수:", data_large.isna().sum().sum())

In [ ]:
# 변수별 평균으로 보간
data_mean = data2.fillna(data2.mean(numeric_only=True))
print("[평균 보간] 결측치 수:", data_mean.isna().sum().sum())

In [ ]:
# 변수별 중앙값으로 보간
data_median = data2.fillna(data2.median(numeric_only=True))
print("[중앙값 보간] 결측치 수:", data_median.isna().sum().sum())

In [ ]:
# 특정 변수 1개를 골라, 대치 방법별 값 비교
row_idx = data2.isna().any(axis=1).idxmax()

nan_cols_all = data2.columns[data2.loc[row_idx].isna()]
target_col = nan_cols_all[0]  # 비교할 변수

compare = pd.DataFrame({
    "원본 (NaN)": [data2.loc[row_idx, target_col]],
    "0 대치": [data_zero.loc[row_idx, target_col]],
    "99999 대치": [data_large.loc[row_idx, target_col]],
    "평균 대치": [data_mean.loc[row_idx, target_col]],
    "중앙값 대치": [data_median.loc[row_idx, target_col]]
}, index=[target_col])

print(f"행 {row_idx}, 변수 '{target_col}' 대치 비교:")
compare

### 1-4) KNN Imputer

> **KNN Imputer**는 결측치가 있는 데이터의 최근접 이웃 k개를 찾아, 해당 이웃들의 값의 평균으로 결측치를 보완함
> - NaN이 포함된 변수를 제외한 나머지 변수로 NaN Euclidean Distance를 계산함
> - 거리를 기반으로 유사한 데이터의 값을 가져오는 합리적인 방법임

In [ ]:
data2_sorted = data2.sort_values('Time')  # 시간순 정렬
X = data2_sorted.drop(columns=['Time', 'Pass/Fail'])  # 센서값만 대치 대상
y = data2_sorted['Pass/Fail']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # 시간 순서 유지
)

cols, idx_train, idx_test = X_train.columns, X_train.index, X_test.index  # 덮어쓰기 전에 저장

imputer = KNNImputer(n_neighbors=5)

X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=cols, index=idx_train)
X_test = pd.DataFrame(imputer.transform(X_test), columns=cols, index=idx_test)

print("[Train] 결측치 수:", X_train.isnull().sum().sum())
print("[Test] 결측치 수:", X_test.isnull().sum().sum())

### 1-5) 시계열 보간법

> 시계열 데이터의 경우 시간 순서를 고려한 보간법이 효과적임
> - **ffill (forward fill)**: 이전 시점의 값으로 채우기
> - **bfill (backward fill)**: 다음 시점의 값으로 채우기
> - **linear**: 위치 기반 선형 보간
> - **time**: 시간 간격에 비례한 선형 보간 (불균등 시간 간격에 적합)

In [ ]:
# [예제 1] 소규모 시계열 데이터로 각 보간법 결과 비교

dates = pd.date_range("2026-07-01", "2026-07-08", freq="D")
values = [10, np.nan, np.nan, 40, 50, np.nan, 60, 70]
ts = pd.Series(values, index=dates, name="값")

demo = pd.DataFrame({
    "원본": ts,
    "ffill": ts.ffill(),
    "bfill": ts.bfill(),
    "linear": ts.interpolate(method="linear"),
    "time": ts.interpolate(method="time")
})
print(demo)
print()
print("잔여 NaN 개수:",
      f"ffill: {ts.ffill().isnull().sum()},",
      f"bfill: {ts.bfill().isnull().sum()},",
      f"linear: {ts.interpolate(method='linear').isnull().sum()},",
      f"time: {ts.interpolate(method='time').isnull().sum()}")

> **ffill+bfill**: 맨 앞 행이 결측치라 ffill만으로는 채워지지 않는 경우에 사용
> - 먼저 ffill로 채울 수 있는 결측치를 채우고
> - ffill로도 채워지지 않는 맨 앞 결측치는 bfill로 보간

In [ ]:
# [예시 2] 소규모 시계열 데이터로 보간 방법별 동작 확인

dates = pd.date_range("2026-07-01", "2026-07-08", freq="D")
values = [np.nan, 10, np.nan, np.nan, 40, 50, np.nan, np.nan]
ts = pd.Series(values, index=dates, name="값")

demo = pd.DataFrame({
    "원본": ts,
    "ffill": ts.ffill(),
    "bfill": ts.bfill(),
    "linear": ts.interpolate(method="linear"),
    "time": ts.interpolate(method="time"),
    "ffill+bfill": ts.ffill().bfill(),
})
print(demo)
print()
print("잔여 NaN 개수:",
      f"ffill: {ts.ffill().isna().sum()},",
      f"bfill: {ts.bfill().isna().sum()},",
      f"linear: {ts.interpolate(method='linear').isna().sum()},",
      f"time: {ts.interpolate(method='time').isna().sum()},",
      f"ffill+bfill: {ts.ffill().bfill().isna().sum()}")

---

## 2) 이상치 처리

> **이상치(Outlier)** 는 데이터의 일반적인 패턴에서 크게 벗어난 값을 의미함

In [ ]:
# 변수별 평균으로 보간한 데이터 생성
data_imputed = data2.fillna(data2.median(numeric_only=True))  # 숫자형 컬럼만 중앙값으로 결측치 대체
print(data_imputed.isna().sum().sum())  # 남은 결측치 수 확인 (0이면 결측치 없음)

### 2-1) Box Plot

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(data_imputed['0'])
plt.title("Variable '0' - Box Plot")
plt.show()

### 2-2) IQR 기반 이상치 제거

In [ ]:
q1 = data_imputed['0'].quantile(0.25)  # 1사분위수 (하위 25%)
q3 = data_imputed['0'].quantile(0.75)  # 3사분위수 (상위 25%)
iqr = q3 - q1  # 사분위 범위 (IQR)

lower_bound = q1 - 1.5 * iqr  # 이상치 판단 하한선
upper_bound = q3 + 1.5 * iqr  # 이상치 판단 상한선

# 하한선~상한선 범위 안에 있는 데이터만 남김 (이상치 제거)
data_cleaned = data_imputed[(data_imputed['1'] >= lower_bound) & (data_imputed['1'] <= upper_bound)]

print("이상치 제거 전:", len(data_imputed))
print("이상치 제거 후:", len(data_cleaned))

In [ ]:
data_cleaned = data_imputed.copy()  # 원본 유지하고 복사본으로 작업

numeric_cols = data_cleaned.select_dtypes(include='number').columns  # 숫자형 컬럼만 선택

for column in numeric_cols:  # 숫자형 컬럼에 대해서만 반복
    q1 = data_cleaned[column].quantile(0.25)  # 1사분위수
    q3 = data_cleaned[column].quantile(0.75)  # 3사분위수
    iqr = q3 - q1  # 사분위 범위 (IQR)

    lower_bound = q1 - 1.5 * iqr  # 이상치 판단 하한선
    upper_bound = q3 + 1.5 * iqr  # 이상치 판단 상한선

    data_cleaned = data_cleaned[(data_cleaned[column] >= lower_bound) & (data_cleaned[column] <= upper_bound)]

print("이상치 제거 전:", len(data_imputed))
print("이상치 제거 후:", len(data_cleaned))

In [ ]:
# 이상치 제거 전후 Box Plot 비교
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(data_imputed['1']); axes[0].set_title("Before")
axes[1].boxplot(data_cleaned['1']); axes[1].set_title("After")
plt.tight_layout()
plt.show()

---

## 3) Vibe Coding 실습

**[과제 1]**

지수는 IQR 기반 이상치 제거 코드를 적용했더니 데이터가 3개만 남는 현상을 발견했습니다.

이 상태로는 실제 분석에 데이터를 전혀 쓸 수 없다고 판단하여 AI와 상의하여 왜 이런 현상이 발생하는지 원인을 파악하고 데이터를 과도하게 잃지 않으면서 이상치를 처리할 수 있는 방법을 찾아보려 합니다. 

AI와의 대화를 통해 원인과 대안을 정리하고 그중 하나를 실제로 적용하여 이상치 제거 전후 데이터 크기를 비교해 보세요.